# 06 — Baselines (Etapa 5)

Todos treinados **só com amostras normais** (regra por janela: anos T−1, T e T+1 sem evento; `train`, T ≤ 2022) e
avaliados com o protocolo (`avaliacao.py`): teste em 2023–2025, prevalência real, IC por bootstrap de blocos.

**Unidade (desde 29/10):** cada método pontua as 49 **células de 320 m** do patch (2 × 2 tokens do TerraMind);
o score do patch é o **máximo** das células. A média das células (≈ média do patch) aparece só para comparar escalas.
Motivo: no gate, a média de 2,24 km diluía eventos pequenos (0,57 no patch vs. 0,84 no token).

| Baseline | Entrada (por célula) | Pergunta |
|---|---|---|
| Espectral → Isolation Forest | médias das bandas, NDVI, NBR + diferença para o histórico | O FM ajuda em relação a features clássicas? |
| Queda de NDVI | NDVI do histórico − NDVI(t) | Referência espectral sem parâmetros |
| TerraMind → cosseno | Z(t) vs média dos 4 trimestres anteriores | Baseline sem parâmetros |
| TerraMind → cosseno sazonal | Z(t) vs mesmo trimestre em t−4 e t−8 | A sazonalidade atrapalha? |
| TerraMind → kNN estático | só Z(t) | **O tempo importa?** |
| TerraMind → kNN / IForest / Deep SVDD | Z(t) − histórico | Qual detector sobre o FM? |
| AlphaEarth (anual) | cosseno e kNN estático por célula-ano | Comparação com outro FM, na mesma célula |

TerraMind em duas configurações: S2 + S1 (principal) e só S2 (para o E5).

**Fonte dos dados:** `FONTE = 'tapajos_sub'` usa a sub-AOI (desenvolvimento). Ela tem só 16 patches de teste e foi
escolhida pelos rótulos, então a avaliação vira um **diagnóstico** nos blocos de treino em 2023–2025 (holdout só
temporal). **Não reportar.** Com o `stream_embeddings.py` concluído (ou parcial), use `FONTE = 'tapajos'` e depois `'norte_ro'`.

In [ ]:
import sys, os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    raise RuntimeError('Fluxo local desde 29/10 (ver "Limitações de infraestrutura" no PLANO). '
                       'No Colab, siga o setup do 03_cubo_sentinel e use /content/cache para os pixels.')
import json, time
import numpy as np
import pandas as pd
import config, utils, rotulos, dados, embeddings
config.make_dirs()
utils.set_seed(config.SEED)

import avaliacao, baselines
FONTE, AOI = 'tapajos_sub', 'tapajos'        # depois: ('tapajos', 'tapajos') e ('norte_ro', 'norte_ro')
# sub-AOI: só 16 patches de teste -> diagnóstico nos blocos de TREINO em 2023–2025 (holdout temporal; não reportar)
EVAL_SPLIT = 'train' if FONTE.endswith('_sub') else ('test' if AOI == 'tapajos' else 'test_gen')
print('avaliação em', EVAL_SPLIT, '(DIAGNÓSTICO, não reportar)' if EVAL_SPLIT == 'train' else '')
run_dir = utils.new_run(f'etapa5_baselines_{FONTE}')

## 1. Amostras e normais de treino

In [ ]:
s, ctx, lab = baselines.load(FONTE, AOI)
print(len(s), 'amostras (patch, trimestre) com histórico de', config.K_HIST, 'trimestres;',
      len(s) * config.N_CELLS, 'células')
display(s.groupby(['split', 'cobertura'])[['treino_normal', 'val_normal']].sum().astype(int))

## 2. Scores

Passada 1: sorteia até 40 mil células normais de treino e 10 mil de validação. Ajusta os detectores. Passada 2: pontua todas as células.
Sub-AOI: minutos. AOI completa: algumas horas em CPU (rode de noite, se preciso).

In [ ]:
t0 = time.time()
scores, info = baselines.run_all(s, ctx)
print(f'{(time.time() - t0) / 60:.1f} min | épocas do Deep SVDD:', {k: len(v) for k, v in info.items()})
pd.concat([s[['patch_id', 'step']], scores], axis=1).to_csv(run_dir / 'scores_trimestrais.csv.gz', index=False)

## 3. Avaliação

**Trimestral:** rótulos com data sub-anual (DETER, MCD64A1). **Anual:** score do ano = máximo dos trimestres, contra
rótulos anuais (todas as fontes, incluindo PRODES, Hansen e MapBiomas). O AlphaEarth só existe na resolução anual.

In [ ]:
def eval_all(score_df, labels, key):
    out = []
    for col in [c for c in score_df.columns if '|' in c]:
        sc = score_df[['patch_id', key]].assign(score=score_df[col].values).dropna(subset=['score'])
        r = avaliacao.evaluate(sc, labels, lab['grade'], key=key, split=EVAL_SPLIT, n_boot=200)
        out.append(r.assign(metodo=col))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

q_scores = pd.concat([s[['patch_id', 'step']], scores], axis=1)
res_q = eval_all(q_scores, lab['trim'], 'step').assign(resolucao='trimestral')
a_scores = baselines.to_annual(s, scores)

In [ ]:
# AlphaEarth anual nas mesmas células de 320 m (Earth Engine, com cache por lote)
import ee
try:
    ee.Initialize(project=config.EE_PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=config.EE_PROJECT_ID)
grid = rotulos.make_grid(AOI)
grid = grid[grid['patch_id'].isin(set(s['patch_id']))]
t0 = time.time()
ae_pids, ae_Z = baselines.alphaearth_cells(grid, FONTE, config.LABELS_DIR / AOI / 'cache')
print(f'AlphaEarth: {len(ae_pids)} patches x 49 células em {(time.time() - t0) / 60:.1f} min')
ae = baselines.alphaearth_scores(ae_pids, ae_Z, lab['anual'], lab['grade'])
a_scores = a_scores.merge(ae, on=['patch_id', 'year'], how='left')
res_a = eval_all(a_scores, lab['anual'], 'year').assign(resolucao='anual')
res = pd.concat([res_q, res_a], ignore_index=True)
res['agregacao'] = np.where(res['metodo'].str.endswith('[média]'), 'média', 'máximo')
res['metodo'] = res['metodo'].str.replace(' [média]', '', regex=False)
res.to_csv(run_dir / 'metricas.csv', index=False)

In [ ]:
cols = ['AUROC', 'AUROC_lo', 'AUROC_hi', 'AUPRC', 'TPR@1%FPR', 'prevalencia', 'n_pos']
main = res[res['agregacao'] == 'máximo']
for resol in ['trimestral', 'anual']:
    t = main[(main['resolucao'] == resol) & (main['tarefa'] == 'todos')].set_index('metodo')[cols]
    print(f'== {resol}, todas as anomalias ({EVAL_SPLIT}, 2023–2025), score do patch = máximo das células')
    display(t.sort_values('AUROC', ascending=False).round(3))
print('== AUROC anual por tipo (máximo das células)')
display(main[main['resolucao'] == 'anual'].pivot(index='metodo', columns='tarefa', values='AUROC').round(3))

## 4. Escala: máximo das células vs. média do patch

Mesma pergunta do gate, agora no protocolo completo: agregar pelo máximo das células recupera eventos pequenos?

In [ ]:
esc = res[res['tarefa'] == 'todos'].pivot_table(index=['resolucao', 'metodo'], columns='agregacao', values='AUROC')
esc['ganho do máximo'] = esc['máximo'] - esc['média']
display(esc.round(3).sort_values('ganho do máximo', ascending=False))

## 5. E2 em miniatura: o tempo importa?

O alerta do first pass foi que o kNN **estático** (só Z(t)) bateu os scores temporais. Aqui a mesma pergunta com TerraMind trimestral.

In [ ]:
import matplotlib.pyplot as plt
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e1e0d9'
CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']      # slots categóricos na ordem fixa
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED, 'xtick.color': MUTED,
                     'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False})

t = main[main['tarefa'] == 'todos'].copy()
t['familia'] = t['metodo'].str.split(' | ', regex=False).str[0]
t['nome'] = t['metodo'].str.split(' | ', regex=False).str[1]
fams = list(dict.fromkeys(t['familia']))
fig, axes = plt.subplots(1, 2, figsize=(12, 0.32 * t['metodo'].nunique() + 1.6), sharex=True)
for ax, resol in zip(axes, ['trimestral', 'anual']):
    d = t[t['resolucao'] == resol].sort_values('AUROC')
    y = np.arange(len(d))
    for f, c in zip(fams, CAT):
        m = (d['familia'] == f).values
        ax.errorbar(d['AUROC'][m], y[m], xerr=[(d['AUROC'] - d['AUROC_lo'])[m], (d['AUROC_hi'] - d['AUROC'])[m]],
                    fmt='o', ms=7, color=c, ecolor=c, elinewidth=1.5, capsize=0, label=f)
    ax.set_yticks(y, d['nome'] + '  (' + d['familia'] + ')', fontsize=8)
    ax.axvline(0.5, color=GRID, lw=1, ls='--'); ax.grid(axis='x', color=GRID, lw=0.5)
    ax.set_title(f'{resol}: todas as anomalias', color=INK); ax.set_xlabel('AUROC (IC 95%)')
handles = {}
for ax in axes:
    for hh, ll in zip(*ax.get_legend_handles_labels()):
        handles.setdefault(ll, hh)
fig.legend(list(handles.values()), list(handles), frameon=False, loc='lower center', ncol=len(handles))
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(run_dir / 'auroc_baselines.png', dpi=130); plt.show()

In [ ]:
(run_dir / 'resumo.json').write_text(json.dumps({
    'fonte': FONTE, 'eval_split': EVAL_SPLIT, 'unidade': 'célula 320 m, patch = máximo', 'n_amostras': len(s),
    'treino_normais': int(s['treino_normal'].sum()), 'val_normais': int(s['val_normal'].sum()),
    'svdd_epocas': {k: len(v) for k, v in info.items()},
    'auroc_todos': main[main['tarefa'] == 'todos'].set_index(['resolucao', 'metodo'])['AUROC'].round(4).unstack(0).to_dict()},
    indent=2, default=str), encoding='utf-8')
print('Salvo em', run_dir)